In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, SymmetricalLogLocator
import seaborn as sns
from IPython.display import display
from matplotlib.ticker import MaxNLocator, StrMethodFormatter
from matplotlib.ticker import MaxNLocator

In [ ]:
# model comparison and whole-career supplement
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output/pnas'
OUTPUT.mkdir(parents=True,exist_ok=True)
CONFIG = json.loads((ROOT/'config.json').read_text())
Y, SEED = CONFIG['last_year'], CONFIG['seed']
WINDOW = 'last_four'
WINDOW_LABELS = {'last_four':f'Final four years ({Y-3}–{Y})', 'career_total':f'Whole career (0–{Y})'}
SELECTED = ['empirical_complete', 'lognormal_grw', 'ar1_s', 'hurdle_ar1_s_p', 'se_hurdle_s']
LABELS = {'empirical_complete':'DBLP empirical', 'lognormal_grw':'GRW', 'ar1_s':'AR(1)',
          'hurdle_ar1_s_p':'Hurdle AR(1)', 'se_hurdle_s':'Self-exciting hurdle'}
palette = sns.color_palette('colorblind')
COLORS = dict(zip(SELECTED, ['#222222', palette[0], palette[1], palette[4], palette[2]]))
LINES = dict(zip(SELECTED, ['-', '--', '-.', ':', '-']))
MARKERS = dict(zip(SELECTED, ['o', 's', '^', 'D', 'v']))
sns.set_theme(style='ticks', context='paper', font_scale=1,
              rc={'font.size':8, 'axes.labelsize':8, 'axes.titlesize':9,
                  'xtick.labelsize':7, 'ytick.labelsize':7, 'legend.fontsize':8,
                  'pdf.fonttype':42, 'ps.fonttype':42, 'savefig.dpi':400})

In [ ]:
qq_windows = pd.read_csv(INPUT/'window_qq_points.csv')
qq_windows = qq_windows.loc[qq_windows.source.isin(SELECTED) & qq_windows.window.isin([WINDOW,'career_total'])].copy()
qq = qq_windows.loc[qq_windows.window.eq(WINDOW)].copy()
lognormality = pd.read_csv(INPUT/'lognormality_summary.csv')
lognormality = lognormality.loc[lognormality.source.isin(SELECTED)].copy()
ranks = pd.read_csv(INPUT/'rank_mixing.csv')
ranks = ranks.loc[ranks.source.isin(SELECTED) & ranks.measure.eq('annual') & ranks.reference.eq('lag_5')].copy()
comparison = pd.read_csv(INPUT/'model_comparison.csv').set_index('source').loc[SELECTED]
cdf_windows = pd.read_csv(INPUT/'distribution_cdf.csv')
cdf_windows = cdf_windows.loc[cdf_windows.source.isin(SELECTED) & cdf_windows.diagnostic.isin([WINDOW,'career_total'])].copy()
cdf_knots = cdf_windows.loc[cdf_windows.diagnostic.eq(WINDOW)].copy()

In [ ]:
# exact ECDFs from the same four-year sums as the QQ; no pooled annual observations
panel = pd.read_csv(INPUT/'simulation/fit/data/panel_complete.csv', index_col='dblp_id')
arrays = {'empirical_complete':panel.to_numpy(dtype=float).T}
for tag in SELECTED[1:]:
    arrays[tag] = np.load(INPUT/'simulation'/tag/'trajectories.npy', mmap_mode='r')
total_frames = {}
for window,start in [(WINDOW,Y-3),('career_total',0)]:
    total_frames[window] = pd.concat([pd.DataFrame({'source':tag, 'total':array[start:Y+1].sum(axis=0)})
                                    for tag,array in arrays.items()], ignore_index=True)

In [ ]:
# primary numeric summary and explicit whole-career supplement; no individual records exported
summary = lognormality.loc[lognormality.window.eq(WINDOW)].set_index('source').loc[SELECTED].copy()
summary['final_zero_fraction'] = comparison.final_zero_fraction
summary['lag_5_rho20'] = comparison.lag_5_rho20
summary['career_total_qq_r2'] = lognormality.loc[lognormality.window.eq('career_total')].set_index('source').qq_r2
summary['label'] = pd.Series(LABELS)
summary.to_csv(OUTPUT/'pnas_model_summary.csv')
qq.to_csv(OUTPUT/'pnas_model_qq_points.csv', index=False)
ranks.to_csv(OUTPUT/'pnas_model_lag5.csv', index=False)
cdf_knots.to_csv(OUTPUT/'pnas_model_ecdf_knots.csv', index=False)
lognormality.loc[lognormality.window.eq('career_total')].to_csv(OUTPUT/'pnas_career_total_summary.csv', index=False)
qq_windows.loc[qq_windows.window.eq('career_total')].to_csv(OUTPUT/'pnas_career_total_qq_points.csv', index=False)
cdf_windows.loc[cdf_windows.diagnostic.eq('career_total')].to_csv(OUTPUT/'pnas_career_total_ecdf_knots.csv', index=False)
display(summary)

In [ ]:
def draw_qq(ax, window=WINDOW):
    points = qq_windows.loc[qq_windows.window.eq(window)]
    for tag in SELECTED:
        data = points.loc[points.source.eq(tag)].sort_values('normal_quantile')
        sns.lineplot(data=data, x='normal_quantile', y='observed_quantile', ax=ax,
                     color=COLORS[tag], linestyle=LINES[tag], marker=MARKERS[tag],
                     markevery=50, markersize=2.5, linewidth=1.1, estimator=None, errorbar=None, legend=False)
    limits = (points.normal_quantile.min(), points.normal_quantile.max())
    ax.plot(limits, limits, color='.6', linewidth=.7, zorder=0)
    ax.set(title=WINDOW_LABELS[window]+' shape', xlabel='Standard normal quantile',
           ylabel='Standardized log-total quantile')
    ax.xaxis.set_major_locator(MaxNLocator(4))
    ax.yaxis.set_major_locator(MaxNLocator(5))
    sns.despine(ax=ax)
def draw_ecdf(ax, window=WINDOW):
    for tag in SELECTED:
        sns.ecdfplot(data=total_frames[window].loc[total_frames[window].source.eq(tag)], x='total', ax=ax,
                     color=COLORS[tag], linestyle=LINES[tag], linewidth=1.2, legend=False)
    # shared scale and full tails; linear segment preserves the mass at zero
    ax.set_xscale('symlog', linthresh=1, linscale=.5, base=10)
    ax.set_xlim(left=0)
    locator = SymmetricalLogLocator(base=10,linthresh=1)
    locator.set_params(numticks=4)
    upper = ax.get_xlim()[1]
    ax.set_xticks([value for value in locator.tick_values(0,upper) if value==0 or 1<value<=upper])
    ax.set(title=WINDOW_LABELS[window]+' scale', xlabel='Four-year total output' if window==WINDOW else 'Career-total output',
           ylabel='Cumulative probability', ylim=(0,1.02), yticks=[0,.25,.5,.75,1])
    ax.text(.98, .05, 'Log scale above 1', transform=ax.transAxes, ha='right', fontsize=6.5)
    sns.despine(ax=ax)
def draw_rank(ax):
    for tag in SELECTED:
        data = ranks.loc[ranks.source.eq(tag)].sort_values('age')
        sns.lineplot(data=data, x='age', y='rho', ax=ax, color=COLORS[tag],
                     linestyle=LINES[tag], marker=MARKERS[tag], markevery=3,
                     markersize=3, linewidth=1.2, estimator=None, errorbar=None, legend=False)
    ax.set(title='Annual rank memory', xlabel='Career year, $t$',
           ylabel=r'Spearman $\rho_S(R_{t-5},R_t)$',
           xlim=(ranks.age.min(),ranks.age.max()), ylim=(-.05,1.05))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=4))
    ax.yaxis.set_major_locator(MaxNLocator(5))
    sns.despine(ax=ax)
def legend_handles():
    return [Line2D([], [], color=COLORS[tag], linestyle=LINES[tag], marker=MARKERS[tag],
                   markersize=3, linewidth=1.2, label=LABELS[tag]) for tag in SELECTED]
def save_figure(fig, stem):
    for extension in ['pdf','png']:
        fig.savefig(OUTPUT/f'{stem}.{extension}', bbox_inches='tight', facecolor='white')

In [ ]:
# primary manuscript composite; shared legend above the panels
fig, axes = plt.subplots(1,3,figsize=(7.1,3.25))
for letter,draw,ax in zip('ABC',[draw_qq,draw_ecdf,draw_rank],axes):
    draw(ax)
    ax.text(-.18,1.09,letter,transform=ax.transAxes,weight='bold',fontsize=10)
fig.legend(handles=legend_handles(),loc='upper center',ncol=3,frameon=False,
           bbox_to_anchor=(.5,1.01),columnspacing=1.4,handlelength=2.8)
fig.subplots_adjust(left=.075,right=.99,bottom=.19,top=.73,wspace=.55)
save_figure(fig,'pnas_dblp_model_comparison')
plt.show()
plt.close(fig)

In [ ]:
# individual panels for alternative manuscript layouts
for name,draw in [('qq',draw_qq),('ecdf',draw_ecdf),('lag5',draw_rank)]:
    fig,ax = plt.subplots(figsize=(3.5,3.3))
    draw(ax)
    fig.legend(handles=legend_handles(),loc='upper center',ncol=2,frameon=False,
               bbox_to_anchor=(.5,1.01),handlelength=2.8)
    fig.subplots_adjust(left=.20,right=.96,bottom=.17,top=.70)
    save_figure(fig,f'pnas_dblp_{name}')
    plt.close(fig)

In [ ]:
# retain the previous whole-career distribution view as an explicit supplement
fig,axes = plt.subplots(1,2,figsize=(7.1,3.25))
draw_qq(axes[0],window='career_total')
draw_ecdf(axes[1],window='career_total')
fig.legend(handles=legend_handles(),loc='upper center',ncol=3,frameon=False,
           bbox_to_anchor=(.5,1.01),handlelength=2.8)
fig.subplots_adjust(left=.09,right=.99,bottom=.19,top=.73,wspace=.38)
save_figure(fig,'pnas_dblp_career_total_supplement')
plt.show()
plt.close(fig)

In [ ]:
# aggregation, zero handling and cohort stated alongside the figure
n_empirical = int(summary.loc['empirical_complete','n_total'])
n_simulated = int(summary.loc['lognormal_grw','n_total'])
caption = (
    r'\textbf{Four-year distributional shape and temporal agreement of DBLP productivity models.} '
    f'Empirical comparisons use {n_empirical:,} complete publication histories over career years '
    f'0--{Y}; each model generates {n_simulated:,} histories. '
    f'Panels A and B use the sum of annual adjusted productivity over the final four career years, '
    f'{Y-3}--{Y}, matching Shockley\'s four-year aggregation. '
    r'(A) Normal QQ curves of positive four-year totals after taking $\log_2$ and standardizing '
    r'each source separately to zero mean and unit sample standard deviation. '
    r'The grey diagonal is the normal reference. Standardization removes location and scale; '
    r'zero totals are excluded only from this panel and their counts are reported separately. '
    r'QQ curves retain up to 401 ordered points, including the extremes; reported QQ $R^2$ uses all positive totals. '
    r'(B) Exact empirical cumulative distribution functions of the same four-year totals, including zeros '
    r'and full tails. The shared output axis is linear from zero to one and logarithmic above one. '
    r'(C) Spearman correlation of annual productivity ranks five years apart, with average ranks for ties. '
    r'The same complete empirical cohort is used at every year. '
    r'Models share colors and line styles across panels. These are in-sample generative checks. '
    r'The self-exciting hurdle model is fitted to complete histories; the other models use available '
    r'adjacent-year observations. Whole-career QQ and ECDF views are retained in a separate supplement.'
)
(OUTPUT/'pnas_dblp_model_comparison_caption.tex').write_text(caption+'\n')
supplement_caption = (
    r'\textbf{Whole-career distributional supplement.} '
    f'QQ and exact ECDF views use totals over career years 0--{Y}. '
    r'Cohorts, standardization, zero handling and display conventions match the primary four-year figure.'
)
(OUTPUT/'pnas_dblp_career_total_supplement_caption.tex').write_text(supplement_caption+'\n')
display(pd.DataFrame({'file':['pnas_dblp_model_comparison.pdf','pnas_dblp_career_total_supplement.pdf',
                             'pnas_model_summary.csv','pnas_career_total_summary.csv'],
                      'purpose':['primary four-year figure','whole-career supplement',
                                 'primary numeric summary','whole-career numeric summary']}))

In [ ]:
# rank mixing and temporal panels
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output/pnas'
OUTPUT.mkdir(parents=True,exist_ok=True)
CONFIG = json.loads((ROOT/'config.json').read_text())
Y, SEED = CONFIG['last_year'], CONFIG['seed']
REFERENCES = ['age_0','lag_3','lag_5']
LABELS = {'age_0':r'$\rho_S(R_0,R_t)$', 'lag_3':r'$\rho_S(R_{t-3},R_t)$', 'lag_5':r'$\rho_S(R_{t-5},R_t)$'}
COLORS = dict(zip(REFERENCES,sns.color_palette('colorblind',3)))
LINES = dict(zip(REFERENCES,['-','--','-.']))
MARKERS = dict(zip(REFERENCES,['o','s','^']))
plt.rcdefaults()
sns.set_theme(style='ticks',context='paper',font_scale=1.15,
              rc={'pdf.fonttype':42,'ps.fonttype':42,'savefig.dpi':300})

In [ ]:
# annual ranks of the fixed complete-career DBLP cohort; zeros retain average ranks
ranks = pd.read_csv(INPUT/'rank_mixing.csv')
ranks = ranks.loc[ranks.source.eq('empirical_complete') & ranks.measure.eq('annual') & ranks.reference.isin(REFERENCES)].copy()
ranks = ranks.sort_values(['reference','age'])
ranks.to_csv(OUTPUT/'dblp_rank_mixing.csv',index=False)

In [ ]:
# pooled empirical follow-up for both publication transitions
MIN_REENTRY_RISK = 20
dropout = pd.read_csv(INPUT/'dropout_by_productivity.csv')
dropout = dropout.loc[dropout.source.eq('empirical_available') & dropout.stage.eq('pooled')].copy()
dropout['decile'] = dropout['bin'] + 1
reentry = pd.read_csv(INPUT/'restart_hazard.csv')
reentry = reentry.loc[reentry.source.eq('empirical_available') & reentry.n_at_risk.ge(MIN_REENTRY_RISK)]

In [ ]:
summary = ranks.groupby('reference',sort=False).agg(start_age=('age','min'),end_age=('age','max'), n=('n','first')).reindex(REFERENCES)
summary['terminal_rho'] = ranks.loc[ranks.age.eq(Y)].set_index('reference').rho
summary['measure'] = 'annual'
summary['cohort'] = 'DBLP complete careers'
summary.to_csv(OUTPUT/'dblp_rank_mixing_summary.csv')
display(summary)

In [ ]:
def plot_rank(ax, legend_size=None):
    for reference in REFERENCES:
        data = ranks.loc[ranks.reference.eq(reference)].sort_values('age')
        sns.lineplot(data=data, x='age', y='rho', ax=ax, color=COLORS[reference],
                     linestyle=LINES[reference], marker=MARKERS[reference],
                     markersize=3.5, markevery=2, linewidth=1.4,
                     estimator=None, errorbar=None, label=LABELS[reference])

    ax.set(xlabel='Career year, $t$', ylabel=r'Spearman $\rho$',
           xlim=(0, Y), ylim=(0, 1.05))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=5))
    ax.set_yticks([0, .25, .5, .75, 1])
    legend = ax.legend(frameon=False, loc='upper right', bbox_to_anchor=(1, 1.07),handlelength=2.2, labelspacing=.25, borderaxespad=.3,fontsize=5.6, markerscale=.8)
    for line in legend.get_lines():
        line.set_linewidth(line.get_linewidth() * .8)
        line.set_markeredgewidth(line.get_markeredgewidth() * .8)


def plot_transition(ax, data, x, y, xlabel, xticks):
    sns.lineplot(data=data, x=x, y=y, ax=ax, estimator=None, errorbar=None,
                 color='.2', marker='o', markersize=3, linewidth=1.15)
    ax.set(xlabel=xlabel, ylabel='', xticks=xticks, ylim=(0,None))
    ax.yaxis.set_major_locator(MaxNLocator(3, steps=[1,2,2.5,5,10]))
    ax.yaxis.set_major_formatter(StrMethodFormatter('{x:g}'))

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 2.5))
plot_rank(ax)
sns.despine(ax=ax)
fig.tight_layout(pad=.35)
for extension in ['pdf', 'png']:
    fig.savefig(OUTPUT/f'dblp_rank_mixing.{extension}', bbox_inches='tight',
                pad_inches=.02, facecolor='white')
plt.show()
plt.close(fig)

In [ ]:
for layout, size in [('wide',(7.1,1.6)), ('column',(3.42,3.45))]:
    with plt.rc_context({'axes.titlesize':8.5,'axes.labelsize':8,
                         'xtick.labelsize':7,'ytick.labelsize':7,
                         'xtick.major.pad':2,'ytick.major.pad':2,
                         'axes.labelpad':3,'axes.titlepad':5,
                             'axes.linewidth':.65,'xtick.major.width':.65,'ytick.major.width':.65,
                             'xtick.major.size':2.5,'ytick.major.size':2.5}):
        fig = plt.figure(figsize=size, layout='constrained')
        if layout == 'wide':
            grid = fig.add_gridspec(1,3,width_ratios=[2,1,1])
            rank_ax, dropout_ax, reentry_ax = (fig.add_subplot(spec) for spec in (grid[0,0],grid[0,1],grid[0,2]))
        else:
            grid = fig.add_gridspec(2,2,height_ratios=[1.65,1])
            rank_ax = fig.add_subplot(grid[0,:])
            dropout_ax, reentry_ax = (fig.add_subplot(spec) for spec in (grid[1,0],grid[1,1]))
        plot_rank(rank_ax, legend_size=7)
        plot_transition(dropout_ax, dropout, 'decile', 'probability', 'Output decile',
                        [1,5,10])
        plot_transition(reentry_ax, reentry, 'duration', 'restart_probability',
                        'Consecutive zero years', reentry.duration)
        dropout_ax.set_ylabel('Next-year probability')
        for ax, label in zip((rank_ax,dropout_ax,reentry_ax),
                             ('A  Rank persistence','B  Dropout','C  Reentry')):
            ax.set_title(label, loc='left', fontweight='bold')
        sns.despine(fig=fig)
        for extension in ['pdf','png']:
            fig.savefig(OUTPUT/f'pnas_dblp_temporal_{layout}.{extension}', facecolor='white')
        plt.show()
        plt.close(fig)

In [ ]:
# AR(1) stage parameters
TASK = Path.cwd().parent
INPUT, OUTPUT = TASK/'input', TASK/'output/pnas'
OUTPUT.mkdir(parents=True,exist_ok=True)
STEM = 'pnas_dblp_ar1_stage_parameters'
plt.rcdefaults()
sns.set_theme(style='ticks', context='paper', palette='colorblind',
              rc={'pdf.fonttype':42, 'axes.labelsize':7.5, 'axes.titlesize':8,
                  'xtick.labelsize':6, 'ytick.labelsize':7, 'axes.linewidth':.6,
                  'xtick.major.size':2.5, 'ytick.major.size':2.5,
                  'xtick.major.width':.6, 'ytick.major.width':.6})
COLORS = sns.color_palette('colorblind')
LEFT, RIGHT = COLORS[0], COLORS[3]

In [ ]:
params = pd.read_csv(INPUT/'simulation/fit/ar1_s/parameters.csv').sort_values('start')
# stationary log-offset mean with stage coefficients held fixed and no zero floor
params['implied_log_mean'] = (params.intercept/(1-params.beta)).where(params.beta.abs().lt(1))
display(params[['stage','intercept','beta','sigma','implied_log_mean']])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(3.42,1.25), sharex=True, layout='constrained')
fig.get_layout_engine().set(w_pad=.025, h_pad=.025, wspace=.035)
panels = [
    ('A  Coefficients', 'intercept', r'$\alpha$', 'beta', r'$\beta$'),
    ('B  Shock and mean', 'sigma', r'Shock SD, $\sigma$', 'implied_log_mean', r'$\alpha/(1-\beta)$'),
]
for ax, (title,left,left_label,right,right_label) in zip(axes,panels):
    twin = ax.twinx()
    sns.lineplot(data=params, x='stage', y=left, ax=ax, color=LEFT,
                 marker='o', markersize=3.5, linewidth=1.15, estimator=None, errorbar=None)
    sns.lineplot(data=params, x='stage', y=right, ax=twin, color=RIGHT,
                 marker='s', markersize=3.5, linewidth=1.15, linestyle='--',
                 estimator=None, errorbar=None)
    ax.set(xlabel='', ylabel=left_label)
    twin.set(ylabel=right_label)
    ax.set_title(title, loc='left', fontweight='bold')
    for side, name, color in [(ax,'left',LEFT), (twin,'right',RIGHT)]:
        side.margins(y=.14)
        side.yaxis.set_major_locator(MaxNLocator(nbins=3, steps=[1,2,2.5,5,10]))
        side.yaxis.label.set_color(color)
        side.yaxis.labelpad = 2
        side.tick_params(axis='y', colors=color, pad=1.5)
        side.spines[name].set_color(color)
    sns.despine(ax=ax, right=True)
    sns.despine(ax=twin, left=True, right=False, bottom=True)
fig.supxlabel('Transition-start years', fontsize=7.5)
for extension in ('pdf','png'):
    fig.savefig(OUTPUT/f'{STEM}.{extension}', dpi=300, facecolor='white')
plt.show()
plt.close(fig)